In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib import colors as cols
import numpy as np
from lib import vaspwfc
from pymatgen.core import Lattice, Structure, Molecule
from pymatgen.electronic_structure.core import Spin, Orbital
from pymatgen.io.ase import Atoms, AseAtomsAdaptor
from pymatgen.io import vasp
import nglview as nv
from ase.neighborlist import natural_cutoffs, NeighborList
import h5py

In [ ]:
from lib import Initio

In [ ]:
class Initio:
    def __init__(self):
        pass

    def read_vasp_file(self, path: str) -> vaspwfc | Structure | bool:
        base_name = os.path.basename(path)
        
        match base_name:
            case "WAVECAR":
                wfc = self.get_wavecar(path)
                return wfc
            case "POSCAR" | "CONTCAR":
                struc = self.get_structure(path)
                return struc
            case _:
                print(f"Could not determine file type of provided file {path}")
                return False

    def get_wavecar(self, path: str, lgamma: bool = True) -> vaspwfc:
        try:
            wfc = vaspwfc(path, lgamma = lgamma)
            return wfc
        except Exception as e:
            print("Error loading the wavecar")
            return False

    def get_structure(self, path: str) -> Structure:
        try:
            structure = Structure.from_file(path)
            return structure
        except Exception as e:
            print("Error loading the structure")
            return False

    def DOS_from_energies(self, eigenenergies: list | np.ndarray = [], gamma = None, sigma = None, energy_range = None, points = None, dE: float = 0.1, weights: list | np.ndarray = []) -> np.ndarray:
        use_weights = False
        
        if isinstance(eigenenergies, list): eigenenergies = np.array(eigenenergies, dtype = float)
        if not isinstance(eigenenergies, np.ndarray): raise TypeError("No valid energy list given")
        
        if isinstance(weights, list): weights = np.array(weights, dtype = float)
        if isinstance(weights, np.ndarray) and len(weights) == len(eigenenergies): use_weights = True

        E_min = np.min(eigenenergies)
        E_max = np.max(eigenenergies)
        
        if isinstance(energy_range, list | np.ndarray):
            energy_range.sort()
            if len(energy_range) > 1:
                E_min = energy_range[0]
                E_max = energy_range[1]
        
        if isinstance(points, int): # Explicit specification of the number of points triggers the energy list to be composed using linspace
            E_list = np.linspace(E_min, E_max, points, dtype = float)
        else: # Use energy spacing instead
            E_list = np.arange(E_min, E_max + dE, dE)
            
        DOS = np.stack([E_list, np.zeros_like(E_list)], dtype = float)
        


        # Use Lorentzian broadening                
        if isinstance(gamma, float) and gamma > 0:
            gamma2 = gamma ** 2
            
            for index, energy in enumerate(E_list):
                en_diff_list = eigenenergies - energy
                en_diff_list2 = en_diff_list ** 2
                
                if use_weights:
                    for eigenenergy_index, delta_E2 in enumerate(en_diff_list2):
                        DOS[1, index] += weights[eigenenergy_index] * gamma / (gamma2 + delta_E2)
                else:
                    for delta_E2 in en_diff_list2:
                        DOS[1, index] += gamma / (gamma2 + delta_E2)
        
        return DOS

    def get_HOMO_LUMO(self, wavecar_object: vaspwfc) -> dict:
        all_bands = wavecar_object._bands
        all_band_occs = wavecar_object._occs
        
        [bands_up, bands_down] = [all_bands[i] for i in range(2)]
        [occs_up, occs_down] = [all_band_occs[i] for i in range(2)]
        
        [bands_up_gamma, bands_down_gamma] = [bands_up[0], bands_down[0]] # Assuming k-point 0 is the Gamma point
        [occs_up_gamma, occs_down_gamma] = [occs_up[0], occs_down[0]]
        
        LUMO_up_index = int(np.where(occs_up_gamma < .5)[0][0])
        HOMO_up_index = LUMO_up_index - 1
        LUMO_down_index = int(np.where(occs_down_gamma < .5)[0][0])
        HOMO_down_index = LUMO_down_index - 1
        
        HOMO_up_energy = float(bands_up_gamma[HOMO_up_index])
        HOMO_down_energy = float(bands_down_gamma[HOMO_down_index])
        LUMO_up_energy = float(bands_up_gamma[LUMO_up_index])
        LUMO_down_energy = float(bands_down_gamma[LUMO_down_index])
        
        return {"HOMO_up_index": HOMO_up_index, "HOMO_down_index": HOMO_down_index, "LUMO_up_index": LUMO_up_index, "LUMO_down_index": LUMO_down_index,
                "HOMO_up_energy": HOMO_up_energy, "HOMO_down_energy": HOMO_down_energy, "LUMO_up_energy": LUMO_up_energy, "LUMO_down_energy": LUMO_down_energy}

    def get_eigenenergies_from_wavecar(self, wavecar_object: vaspwfc) -> dict:
        all_bands = wavecar_object._bands
        all_band_occs = wavecar_object._occs
        
        [bands_up, bands_down] = [all_bands[i] for i in range(2)]
        [occs_up, occs_down] = [all_band_occs[i] for i in range(2)]
        
        [bands_up_gamma, bands_down_gamma] = [bands_up[0], bands_down[0]] # Assuming k-point 0 is the Gamma point
        [occs_up_gamma, occs_down_gamma] = [occs_up[0], occs_down[0]]
        
        return {"energies": {"spin up": bands_up_gamma, "spin down": bands_down_gamma}, "occupations": {"spin up": occs_up_gamma, "spin down": occs_down_gamma}}

    def spin_and_occupation_resolved_DOS(self, wavecar_object: vaspwfc, *args, **kwargs) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        weights = kwargs.pop("weights", None)
        
        energy_dict = self.get_eigenenergies_from_wavecar(wavecar_object)
        [bands_up_gamma, bands_down_gamma] = [energy_dict["energies"][spin] for spin in ["spin up", "spin down"]]
        [occs_up_gamma, occs_down_gamma] = [energy_dict["occupations"][spin] for spin in ["spin up", "spin down"]]

        LDOS_up_occ = self.DOS_from_energies(bands_up_gamma, weights = occs_up_gamma, *args, **kwargs)
        LDOS_up_unocc = self.DOS_from_energies(bands_up_gamma, weights = 1 - occs_up_gamma, *args, **kwargs)
        LDOS_down_occ = self.DOS_from_energies(bands_down_gamma, weights = occs_down_gamma, *args, **kwargs)
        LDOS_down_unocc = self.DOS_from_energies(bands_down_gamma, weights = 1 - occs_down_gamma, *args, **kwargs)
        
        return (LDOS_up_occ, LDOS_up_unocc, LDOS_down_occ, LDOS_down_unocc)

    def DOS_plot(self, wavecar_object: vaspwfc, *args, **kwargs) -> plt.Figure:
        colors = kwargs.pop("colors", None)
        
        # No colors given. Use defaults
        if not isinstance(colors, list) or len(colors) < 2: colors = ["#A00000", "#0000A0"]
        # Invalid colors given. Use defaults
        if not cols.is_color_like(colors[0]): colors = ["#A00000", "#0000A0"]

        col_up_occ = list(cols.to_rgb(colors[0]))
        col_up_unocc = [.5 + .5 * channel for channel in col_up_occ]
        col_down_occ = list(cols.to_rgb(colors[1]))
        col_down_unocc = [.5 + .5 * channel for channel in col_down_occ]
        
        (LDOS_up_occ, LDOS_up_unocc, LDOS_down_occ, LDOS_down_unocc) = self.spin_and_occupation_resolved_DOS(wavecar_object, *args, **kwargs)
        
        fig, ax = plt.subplots()        
        ax.fill_betweenx(LDOS_up_occ[0], LDOS_up_occ[1], color = col_up_occ)
        ax.fill_betweenx(LDOS_up_unocc[0], LDOS_up_unocc[1], color = col_up_unocc)
        ax.fill_betweenx(LDOS_down_occ[0], -LDOS_down_occ[1], color = col_down_occ)
        ax.fill_betweenx(LDOS_down_unocc[0], -LDOS_down_unocc[1], color = col_down_unocc)
        
        return fig



In [ ]:
WS2_folder = "C:\\DFT\\WS2"

calculation_folders = {name: os.path.join(WS2_folder, name) for name in ["primitive", "Vac_S_0", "supercell", "Co_S_0", "Co_S_-1", "Co_S_+1"]}

In [ ]:
calculation_name = "Co_S_0"
os.chdir(calculation_folders[calculation_name])
print(f"Analyzing files in calculation folder {os.getcwd()}")

initio = Initio()
[struc, wfc] = [initio.read_vasp_file(file_name) for file_name in ["./CONTCAR", "./WAVECAR"]]

view = nv.show_pymatgen(struc)
view.add_representation("unit_cell")
view

In [ ]:
view.clear_representations()
view.add_representation("ball+stick", radius = .5, bond_cutoff = 10)


In [ ]:
HL_dict = initio.get_HOMO_LUMO(wfc)
[HOMO_up_index, HOMO_down_index] = [HL_dict.get(attribute) for attribute in ["HOMO_up_index", "HOMO_down_index"]]
print(f"Spin up HOMO / LUMO band indices: [{HOMO_up_index}, {HOMO_up_index + 1}]\nSpin down HOMO / LUMO band indices: [{HOMO_down_index}, {HOMO_down_index + 1}]")

figure = initio.DOS_plot(wfc, energy_range = [-3.6, 0], dE = .01, gamma = .03)
plt.show()

In [ ]:
tip_height_Angstrom = 1.5
band_indices = range(HOMO_up_index - 5, HOMO_up_index + 5)



orb_index = HOMO_up_index
wfn = wfc.wfc_r(1, 1, orb_index)
lat_vec = wfc._Acell
wfn_pixels = wfn.shape
wfn_array = np.zeros(shape = (2, len(band_indices), wfn_pixels[0], wfn_pixels[1]))
Angstrom_per_px = [float(lat_vec[dim, dim] / wfn_pixels[dim]) for dim in range(3)]
surface_height_Angstrom = float(np.max(struc.cart_coords[:, 2]))
abs_height_Angstrom = surface_height_Angstrom + tip_height_Angstrom
height_slice_index = int(round(abs_height_Angstrom / Angstrom_per_px[2]))

for spin in range(2):
    for iteration, orb_index in enumerate(band_indices):
        print(f"Iteration {iteration}: analyzing single-particle wavefunction number {orb_index} for spin channel {spin}")
        wfn = wfc.wfc_r(spin + 1, 1, orb_index)
        #wfc.save2vesta(wfn, poscar = "./CONTCAR", prefix = f"spin_{spin}_orb_{orb_index}")

        wfn_slice = wfn[:, :, height_slice_index]
        wfn_array[spin, iteration] = wfn_slice

        orb_pos = np.clip(wfn_slice, a_min = 0, a_max = 1) ** 2
        orb_neg = np.clip(wfn_slice, a_min = -1, a_max = 0) ** 2
        plt.imsave(f"spin_{spin}_orb_{orb_index}.png", orb_pos + orb_neg, cmap = "gray")

energy_dict = initio.get_eigenenergies_from_wavecar(wfc)

with h5py.File("./wfns_at_2A.h5", "w") as f:
    wfns_ds = f.create_dataset("wavefunctions", data = wfn_array)
    f.create_dataset("lattice_vectors", data = lat_vec)
    f.create_dataset("orbital_energies", data = np.stack([energy_dict["energies"]["spin up"], energy_dict["energies"]["spin down"]]))
    f.create_dataset("orbital_occupations", data = np.stack([energy_dict["occupations"]["spin up"], energy_dict["occupations"]["spin down"]]))
    f.create_dataset("orbital_numbers", data = band_indices)
